#1. Import Libraries

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, accuracy_score
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
import time

# 2. Load Data

In [2]:
# Load Data
df_stress = pd.read_csv('stress_test_eval_set_15k.csv', engine='python', on_bad_lines='skip')
df_stratified = pd.read_csv('stratified_evaluation_set_8k.csv', engine='python', on_bad_lines='skip')
labels = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
print('Stress DF')
display(df_stress.head())

print('\nStratified DF')
display(df_stratified.head())

Stress DF


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,is_clean
0,703597fe86bade85,", which are then available for download online",0,0,0,0,0,0,1
1,18d027b550b1cc35,"""Obama is NOT """"African American"""" you jackass...",1,0,1,0,1,0,0
2,c6558504c4ded75f,"""==AWARD OF THE PURPLE HEAD TO FISHERQUEEN QNT...",0,0,0,0,0,0,1
3,85a821ce55b3dd26,GO FUCK YOURSELF\n\nALLAHU AKBAR!!!!!!,1,1,1,0,1,1,0
4,5dcd3da9070aa506,How are you MudkipsPlz? Or should I call you C...,1,0,0,0,0,0,0



Stratified DF


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,word_count,power_label
0,d2ad247a8f0e9e86,"I don't aim to debate or complain, Deskana, an...",0,0,0,0,0,0,141,0
1,99f6cbade5ff8d45,History is part of human civilization that has...,0,0,0,0,0,0,348,0
2,c2c89dc2fffdd3fd,Needs Cleanup\nThis article needs work. It als...,0,0,0,0,0,0,32,0
3,c92b1d6a6a970da7,IGN Rankings overused \n\nAre these rankings r...,0,0,0,0,0,0,26,0
4,e985354ca9a3f9b2,"""|benefits]]. If you edit without a username, ...",0,0,0,0,0,0,324,0


# 3. Functions to Set up Model and Run Inference

In [3]:
def setup_classifier(model_name):
    """Sets up the architectural base model with a 6-label head."""
    # Master Seed for Reproducibility (Rubric Requirement)
    torch.manual_seed(42)

    device = 0 if torch.cuda.is_available() else -1

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # num_labels=6 matches: toxic, severe_toxic, obscene, threat, insult, identity_hate
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)
    model.eval()

    classifier = pipeline(
        "text-classification",
        model=model,
        tokenizer=tokenizer,
        device=device,
        top_k=None, # Returns all 6 scores
        truncation=True,    # Cut at 512 TOKENS
        max_length=512,      # Define maximum token BERT can handle
        function_to_apply="sigmoid"
    )

    print(model.config.id2label)

    return classifier

In [4]:
def run_inference(df, text_column, classifier, batch_size=32):
    """
    Processes the dataframe using the Hugging Face Pipeline.
    The pipeline handles tokenization, device management, and sigmoid automatically.
    """
    # 1. Convert the column to a list for the pipeline
    texts = df[text_column].astype(str).tolist()

    # 2. Run the pipeline
    start_time = time.perf_counter()

    raw_results = classifier(texts, batch_size=batch_size, truncation=True, max_length=512)

    end_time = time.perf_counter()
    elapsed_time = end_time - start_time

    # 3. Format the results into a NumPy array to match your previous 'raw_scores' format
    # This transforms [{'label': 'LABEL_0', 'score': 0.5}, ...] into a flat list of scores
    all_scores = []
    for result in raw_results:
        # We sort by label name or index to ensure columns stay consistent
        # For bert-base-uncased, labels are usually 'LABEL_0', 'LABEL_1', etc.
        sorted_scores = [item['score'] for item in sorted(result, key=lambda x: x['label'])]
        all_scores.append(sorted_scores)

    return np.array(all_scores), elapsed_time

# 4. BERT Cased - Stress Test

In [7]:
model_name = "google-bert/bert-base-cased"

global_start_time = time.perf_counter()

# 1. Setup

setup_start = time.perf_counter()
model = setup_classifier(model_name)
setup_duration = time.perf_counter() - setup_start

# 2. Run Inference

raw_scores_stress, duration_stress = run_inference(df_stress, 'comment_text', model, batch_size=32)

n_samples = len(df_stress)
latency_per_sample = (duration_stress / n_samples) * 1000  # in milliseconds
throughput = n_samples / duration_stress                  # comments per second

print(f"Total Time: {duration_stress:.2f} seconds")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Avg Latency: {latency_per_sample:.2f} ms/sample")

# 3. Map to DataFrame
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
model_score_cols = [f"{model_name}_score_{name}" for name in label_names]

df_stress[model_score_cols] = raw_scores_stress

model_pred_cols = [f"{model_name}_pred_{name}" for name in label_names]
df_stress[model_pred_cols] = (df_stress[model_score_cols] >= 0.5).astype(int)

y_stress_true = df_stress[label_names].values
y_stress_pred = df_stress[model_pred_cols].values

# Preview the specific predictions for Stress Test
print("\nSample Predictions for Stress Test:")
print(df_stress[['comment_text'] + model_pred_cols].head())

total_wall_time = time.perf_counter() - global_start_time

print("="*50)
print(f"{model_name} (Stress) TIME REPORT")
print("="*50)
print(f"Total Time: {total_wall_time:.2f} seconds")
print(f"Total Model Setup: {setup_duration:.2f} seconds")
print(f"Total Model Inference: {duration_stress:.2f} seconds")
print("="*50)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5'}
Total Time: 275.27 seconds
Throughput: 39.26 samples/sec
Avg Latency: 25.47 ms/sample

Sample Predictions for Stress Test:
                                        comment_text  \
0     , which are then available for download online   
1  "Obama is NOT ""African American"" you jackass...   
2  "==AWARD OF THE PURPLE HEAD TO FISHERQUEEN QNT...   
3             GO FUCK YOURSELF\n\nALLAHU AKBAR!!!!!!   
4  How are you MudkipsPlz? Or should I call you C...   

   google-bert/bert-base-cased_pred_toxic  \
0                                       0   
1                                       0   
2                                       0   
3                                       1   
4                                       1   

   google-bert/bert-base-cased_pred_severe_toxic  \
0                                              1   
1                                              1   
2                           

In [ ]:
bert_cased_stress_report_dict = classification_report(
    y_stress_true,
    y_stress_pred,
    target_names=label_names,
    zero_division=0,
    output_dict=True
)

stress_accuracy = accuracy_score(y_stress_true, y_stress_pred)

bert_cased_stress_report_df = pd.DataFrame(bert_cased_stress_report_dict).transpose()
bert_cased_stress_report_df.loc['accuracy'] = [np.nan, np.nan, stress_accuracy, y_stress_true.shape[0]]

print(f"\nReport for {model_name} (Stress Test):")
print(f"Global Accuracy: {stress_accuracy:.4f}")
display(bert_cased_stress_report_df)

# 5. BERT Cased - Stratified Test

In [5]:
model_name = "google-bert/bert-base-cased"

global_start_time = time.perf_counter()

# 1. Setup

setup_start = time.perf_counter()
model = setup_classifier(model_name)
setup_duration = time.perf_counter() - setup_start

# 2. Run Inference

raw_scores_stratified, duration_stratified = run_inference(df_stratified,'comment_text',model,batch_size=32)

n_samples = len(df_stratified)
latency_per_sample = (duration_stratified / n_samples) * 1000  # in milliseconds
throughput = n_samples / duration_stratified                  # comments per second

print(f"Total Time: {duration_stratified:.2f} seconds")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Avg Latency: {latency_per_sample:.2f} ms/sample")

# 3. Map to DataFrame
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
model_score_cols = [f"{model_name}_score_{name}" for name in label_names]

df_stratified[model_score_cols] = raw_scores_stratified

model_pred_cols = [f"{model_name}_pred_{name}" for name in label_names]
df_stratified[model_pred_cols] = (df_stratified[model_score_cols] >= 0.5).astype(int)

y_stratified_true = df_stratified[label_names].values
y_stratified_pred = df_stratified[model_pred_cols].values

# Preview the specific predictions for Stratified Test
print("\nSample Predictions for Stratified Test:")
print(df_stratified[['comment_text'] + model_pred_cols].head())

total_wall_time = time.perf_counter() - global_start_time

print("="*50)
print(f"{model_name} (Stratified) TIME REPORT")
print("="*50)
print(f"Total Time: {total_wall_time:.2f} seconds")
print(f"Total Model Setup: {setup_duration:.2f} seconds")
print(f"Total Model Inference: {duration_stratified:.2f} seconds")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3', 4: 'LABEL_4', 5: 'LABEL_5'}
Total Time: 4294.74 seconds
Throughput: 35.29 samples/sec
Avg Latency: 28.34 ms/sample

Sample Predictions for Stratified Test:
                                        comment_text  \
0  I don't aim to debate or complain, Deskana, an...   
1  History is part of human civilization that has...   
2  Needs Cleanup\nThis article needs work. It als...   
3  IGN Rankings overused \n\nAre these rankings r...   
4  "|benefits]]. If you edit without a username, ...   

   google-bert/bert-base-cased_pred_toxic  \
0                                       1   
1                                       1   
2                                       1   
3                                       1   
4                                       1   

   google-bert/bert-base-cased_pred_severe_toxic  \
0                                              1   
1                                              1   
2                      

NameError: name 'duration_stress' is not defined

In [7]:
bert_cased_stratified_report_dict = classification_report(
    y_stratified_true,
    y_stratified_pred,
    target_names=label_names,
    zero_division=0,
    output_dict=True
)

bert_cased_stratified_report_df = pd.DataFrame(bert_cased_stratified_report_dict).transpose()
bert_cased_stratified_report_df

,precision,recall,f1-score,support
toxic,0.120476,0.685164,0.204919,14525.0
severe_toxic,0.009976,0.999339,0.019754,1513.0
obscene,0.057258,0.116870,0.076860,8026.0
threat,0.002958,0.991150,0.005898,452.0
insult,0.000000,0.000000,0.000000,7481.0
identity_hate,0.009917,0.126687,0.018394,1334.0
micro avg,0.031067,0.390597,0.057557,33331.0
macro avg,0.033431,0.486535,0.054304,33331.0
weighted avg,0.067178,0.390597,0.109520,33331.0
samples avg,0.027845,0.044759,0.031635,33331.0


# 6. Toxic Bert - Stress Test

In [6]:
model_name = "unitary/toxic-bert"

global_start_time = time.perf_counter()

# 1. Setup

setup_start = time.perf_counter()
model = setup_classifier(model_name)
setup_duration = time.perf_counter() - setup_start

# 2. Run Inference

raw_scores_stress, duration_stress = run_inference(df_stress, 'comment_text', model, batch_size=32)

n_samples = len(df_stress)
latency_per_sample = (duration_stress / n_samples) * 1000  # in milliseconds
throughput = n_samples / duration_stress                  # comments per second

print(f"Total Time: {duration_stress:.2f} seconds")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Avg Latency: {latency_per_sample:.2f} ms/sample")

# 3. Map to DataFrame
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
model_score_cols = [f"{model_name}_score_{name}" for name in label_names]

df_stress[model_score_cols] = raw_scores_stress

model_pred_cols = [f"{model_name}_pred_{name}" for name in label_names]
df_stress[model_pred_cols] = (df_stress[model_score_cols] >= 0.5).astype(int)

y_stress_true = df_stress[label_names].values
y_stress_pred = df_stress[model_pred_cols].values

# Preview the specific predictions for Stress Test
print("\nSample Predictions for Stress Test:")
print(df_stress[['comment_text'] + model_pred_cols].head())

total_wall_time = time.perf_counter() - global_start_time

print("="*50)
print(f"{model_name} (Stress) TIME REPORT")
print("="*50)
print(f"Total Time: {total_wall_time:.2f} seconds")
print(f"Total Model Setup: {setup_duration:.2f} seconds")
print(f"Total Model Inference: {duration_stress:.2f} seconds")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}
Total Time: 265.51 seconds
Throughput: 40.70 samples/sec
Avg Latency: 24.57 ms/sample

Sample Predictions for Stress Test:
                                        comment_text  \
0     , which are then available for download online   
1  "Obama is NOT ""African American"" you jackass...   
2  "==AWARD OF THE PURPLE HEAD TO FISHERQUEEN QNT...   
3             GO FUCK YOURSELF\n\nALLAHU AKBAR!!!!!!   
4  How are you MudkipsPlz? Or should I call you C...   

   unitary/toxic-bert_pred_toxic  unitary/toxic-bert_pred_severe_toxic  \
0                              0                                     0   
1                              0                                     1   
2                              0                                     0   
3                              0                                     0   
4                              0                                     0   

   

In [ ]:
toxic_bert_stress_report_dict = classification_report(
    y_stress_true,
    y_stress_pred,
    target_names=label_names,
    zero_division=0,
    output_dict=True
)

toxic_bert_stress_report_df = pd.DataFrame(toxic_bert_stress_report_dict).transpose()
toxic_bert_stress_report_df

# 7. Toxic Bert - Stratified Test

In [ ]:
model_name = "unitary/toxic-bert"

global_start_time = time.perf_counter()

# 1. Setup

setup_start = time.perf_counter()
model = setup_classifier(model_name)
setup_duration = time.perf_counter() - setup_start

# 2. Run Inference

raw_scores_stratified, duration_stratified = run_inference(df_stratified,'comment_text',model,batch_size=32)

n_samples = len(df_stratified)
latency_per_sample = (duration_stratified / n_samples) * 1000  # in milliseconds
throughput = n_samples / duration_stratified                  # comments per second

print(f"Total Time: {duration_stratified:.2f} seconds")
print(f"Throughput: {throughput:.2f} samples/sec")
print(f"Avg Latency: {latency_per_sample:.2f} ms/sample")

# 3. Map to DataFrame
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
model_score_cols = [f"{model_name}_score_{name}" for name in label_names]

df_stratified[model_score_cols] = raw_scores_stratified

model_pred_cols = [f"{model_name}_pred_{name}" for name in label_names]
df_stratified[model_pred_cols] = (df_stratified[model_score_cols] >= 0.5).astype(int)

y_stratified_true = df_stratified[label_names].values
y_stratified_pred = df_stratified[model_pred_cols].values

# Preview the specific predictions for Stratified Test
print("\nSample Predictions for Stratified Test:")
print(df_stratified[['comment_text'] + model_pred_cols].head())

total_wall_time = time.perf_counter() - global_start_time

print("="*50)
print(f"{model_name} (Stratified) TIME REPORT")
print("="*50)
print(f"Total Time: {total_wall_time:.2f} seconds")
print(f"Total Model Setup: {setup_duration:.2f} seconds")
print(f"Total Model Inference: {duration_stratified:.2f} seconds")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{0: 'toxic', 1: 'severe_toxic', 2: 'obscene', 3: 'threat', 4: 'insult', 5: 'identity_hate'}
Total Time: 265.51 seconds
Throughput: 40.70 samples/sec
Avg Latency: 24.57 ms/sample

Sample Predictions for Stress Test:
                                        comment_text  \
0     , which are then available for download online   
1  "Obama is NOT ""African American"" you jackass...   
2  "==AWARD OF THE PURPLE HEAD TO FISHERQUEEN QNT...   
3             GO FUCK YOURSELF\n\nALLAHU AKBAR!!!!!!   
4  How are you MudkipsPlz? Or should I call you C...   

   unitary/toxic-bert_pred_toxic  unitary/toxic-bert_pred_severe_toxic  \
0                              0                                     0   
1                              0                                     1   
2                              0                                     0   
3                              0                                     0   
4                              0                                     0   

   

In [ ]:
toxic_bert_stratified_report_dict = classification_report(
    y_stratified_true,
    y_stratified_pred,
    target_names=label_names,
    zero_division=0,
    output_dict=True
)

toxic_bert_stratified_report_df = pd.DataFrame(toxic_bert_stratified_report_dict).transpose()
toxic_bert_stratified_report_df